In [1]:
import os

os.environ['HF_HOME'] = '/research/haider/.cache/hf_home'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/research/haider/.cache/hf'
os.environ['UNIBENCH_HUB'] = '/research/haider/.cache/unibench'
os.environ['TORCH_HOME'] = '/research/haider/.cache/torch'

In [2]:
import pandas as pd

from unibench.benchmarks_zoo.registry import list_benchmarks
from unibench.models_zoo.registry import list_models
from unibench.output import OutputHandler
import seaborn as sns
import matplotlib.pyplot as plt

# models = list_models('vllm') + ['siglip2_so400_16_512']
models = list_models('vllm')

benchmarks = list_benchmarks()
benchmarks.remove('bivlc')

outputhandler = OutputHandler(output_dir='/home/haltahan6/unibench/outputs', download_all_precomputed=False)

outputhandler.load_all_csv(
    model_name=models,
    benchmark_name=benchmarks,
)

results = outputhandler.query(**{"benchmark_name": benchmarks, "model_name": models})
from unibench.common_utils.utils import get_model_mappings
model_mappings = get_model_mappings('name')
results['model_name'] = results["model_name"].map(model_mappings)

from unibench.common_utils.utils import get_benchmark_mappings
benchmark_mappings = get_benchmark_mappings("capability")
results["capability"] = results["benchmark_name"].map(benchmark_mappings)
benchmark_mappings = get_benchmark_mappings("benchmark_type")
results["benchmark_type"] = results["benchmark_name"].map(benchmark_mappings)
benchmark_mappings = get_benchmark_mappings("num_classes")
results["num_classes"] = results["benchmark_name"].map(benchmark_mappings)

results = results[(results['task_name'] == 'multi_choice_classification') | (results['task_name'] == 'multi_choice_relation')]

/home/haltahan6/anaconda3/envs/vllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


File not found:  /home/haltahan6/unibench/outputs/chameleon_30b/sugarcrepe.f
File not found:  /home/haltahan6/unibench/outputs/chameleon_30b/vg_attribution.f
File not found:  /home/haltahan6/unibench/outputs/chameleon_30b/vg_relation.f
File not found:  /home/haltahan6/unibench/outputs/chameleon_30b/winoground.f
File not found:  /home/haltahan6/unibench/outputs/llama_3_2_90b_vision_instruct/caltech101.f
File not found:  /home/haltahan6/unibench/outputs/llama_3_2_90b_vision_instruct/cars.f
File not found:  /home/haltahan6/unibench/outputs/llama_3_2_90b_vision_instruct/cifar10.f
File not found:  /home/haltahan6/unibench/outputs/llama_3_2_90b_vision_instruct/cifar100.f
File not found:  /home/haltahan6/unibench/outputs/llama_3_2_90b_vision_instruct/clevr_count.f
File not found:  /home/haltahan6/unibench/outputs/llama_3_2_90b_vision_instruct/clevr_distance.f
File not found:  /home/haltahan6/unibench/outputs/llama_3_2_90b_vision_instruct/coco_order.f
File not found:  /home/haltahan6/unibench/

In [ ]:
import transformers
import torch

model_id = "meta-llama/Llama-3.3-70B-Instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)


Loading checkpoint shards: 100%|██████████| 8/8 [00:08<00:00,  1.09s/it]


In [6]:
import os

def validate_output_batch(prompts, model_outputs, batch_size=8):
    """Batch validation using Qwen2.5 to determine if model outputs match prompts"""
    all_results = []
    
    for i in tqdm(range(0, len(prompts), batch_size), desc="Validating outputs"):
        batch_prompts = prompts[i:i+batch_size]
        batch_outputs = model_outputs[i:i+batch_size]
        
        # Create validation prompts for the batch
        validation_prompts = []
        for prompt, output in zip(batch_prompts, batch_outputs):
            validation_prompt = f"""Given the following prompt and model output, determine if the model output is a valid response to the question asked in the prompt. Respond with only "VALID" or "INVALID".

            Prompt: {prompt}

            Model Output: {output}

            Is this a valid response?"""
            validation_prompts.append(validation_prompt)
        
        # Prepare batch inputs
        messages_batch = [[{"role": "user", "content": vp}] for vp in validation_prompts]
        texts = [tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) 
                for msgs in messages_batch]
        
        # Tokenize batch
        model_inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(model.device)
        
        # Generate responses
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=10,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Decode responses
        responses = tokenizer.batch_decode(
            generated_ids[:, model_inputs.input_ids.shape[-1]:], 
            skip_special_tokens=True
        )
        
        # Process results
        batch_results = [not ("INVALID" in response.upper()) for response in responses]
        all_results.extend(batch_results)
    
    return all_results

# Create a directory to save results
results_dir = "validation_results"
os.makedirs(results_dir, exist_ok=True)

for model_name in results['model_name'].unique():
    print(f"Processing model: {model_name}")
    model_results = results[results['model_name'] == model_name]

    # Extract prompts and outputs
    prompts = model_results['prompt'].tolist()
    model_outputs = model_results['model_output'].tolist()

    # Perform batch validation
    print(f"Starting validation of {len(prompts)} samples...")
    validation_results = validate_output_batch(prompts, model_outputs, batch_size=8)
    
    res = pd.DataFrame()

    res['prompt'] = prompts
    res['model_output'] = model_outputs
    res['output_valid'] = validation_results
    res['model_name'] = model_name
    res['benchmark_name'] = model_results['benchmark_name'].values
    res['task_name'] = model_results['task_name'].values
    

    print(f"Validation complete. Valid outputs: {sum(validation_results)}/{len(validation_results)} ({sum(validation_results)/len(validation_results)*100:.2f}%)")
    
    # Save results to CSV
    csv_filename = f"{results_dir}/{model_name.replace(' ', '_')}_validation_results.csv"
    res.to_csv(csv_filename, index=False)
    print(f"Results saved to: {csv_filename}")
    

Processing model: Aya Vision 32B
Starting validation of 240637 samples...


Validating outputs:   0%|          | 0/30080 [00:58<?, ?it/s]


KeyboardInterrupt: 